# 13 · Checks for the numbers quoted on the website

The main results come from notebooks 01–12. A few numbers on the website are simple calculations on their saved outputs
(counts, shares, a correlation). This notebook recomputes each of them so that **every number on the website can be checked
in the repository**. It fits no new models and uses only files written by earlier notebooks:

* `data/processed/train.csv`, `test.csv` (notebook 01)
* `data/processed/oof_scores.npz` (notebook 04: out-of-fold scores of the Reference models)
* `data/processed/test_predictions.csv` (notebook 11: test-set PDs, decisions and suggested rates)

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))
import numpy as np
import pandas as pd
from src.credit import DATA, best_cutoff

train = pd.read_csv(DATA / "train.csv")
test = pd.read_csv(DATA / "test.csv")
full = pd.concat([train, test], ignore_index=True)
print(f"cleaned loans: {len(full):,} (train {len(train):,}, test {len(test):,})")

cleaned loans: 32,409 (train 25,927, test 6,482)


## 1. Loans by grade (Overview, Data and Result analysis tabs)

Calculated from the data, not predicted: all cleaned loans that have a recorded interest rate. Also the share of applicants
with a past default in each grade ("grades A and B contain no applicant with a past default").

In [2]:
rated = full[full["loan_int_rate"].notna()]
print(f"loans with a recorded rate: {len(rated):,}  (overall default rate {rated['loan_status'].mean():.1%})")
g = rated.groupby("loan_grade").agg(loans=("loan_status", "size"),
                                    average_rate=("loan_int_rate", "mean"),
                                    default_rate=("loan_status", "mean"))
g["share_of_loans"] = g["loans"] / len(rated)
g["past_default_share"] = full.groupby("loan_grade")["cb_person_default_on_file"].apply(lambda s: (s == "Y").mean())
g.style.format({"loans": "{:,}", "average_rate": "{:.1f}%", "default_rate": "{:.0%}",
                "share_of_loans": "{:.1%}", "past_default_share": "{:.0%}"})

loans with a recorded rate: 29,315  (overall default rate 22.0%)


,loans,average_rate,default_rate,share_of_loans,past_default_share
loan_grade,,,,,
A,"9,712",7.3%,10%,33.1%,0%
B,"9,335",11.0%,16%,31.8%,0%
C,"5,807",13.5%,21%,19.8%,50%
D,"3,308",15.4%,59%,11.3%,52%
E,880,17.0%,65%,3.0%,48%
F,214,18.6%,70%,0.7%,46%
G,59,20.3%,98%,0.2%,56%


## 2. A rule without exceptions (Result analysis findings 2 and 7)

The depth-3 tree (notebook 03) finds that renters who borrow more than about 31% of their income all defaulted.

In [3]:
rent_high = train[(train["person_home_ownership"] == "RENT") & (train["loan_percent_income"] > 0.30)]
print(f"training applicants who rent and borrow more than 30% of income: {len(rent_high):,} "
      f"({len(rent_high) / len(train):.1%} of the training set)")
print(f"share of them who defaulted: {rent_high['loan_status'].mean():.1%}")

training applicants who rent and borrow more than 30% of income: 1,870 (7.2% of the training set)
share of them who defaulted: 100.0%


## 3. SMOTE only moves the cutoff (Result analysis finding 4)

Out-of-fold scores of the tuned logistic regressions with and without SMOTE (Reference setting, saved by notebook 04). A rank correlation near 100% means both models put applicants in the same order; only the best cutoff differs.

In [4]:
oof = np.load(DATA / "oof_scores.npz")
y_tr = train["loan_status"].to_numpy()
a, b = oof["LogReg (no SMOTE)"], oof["LogReg + SMOTE"]
from scipy.stats import spearmanr
print(f"rank correlation of the two score sets (same ordering of applicants = 100%): {spearmanr(a, b)[0]:.1%}")
print(f"F1-best cutoff without SMOTE: {best_cutoff(a, pd.Series(y_tr)):.2f}")
print(f"F1-best cutoff with SMOTE:    {best_cutoff(b, pd.Series(y_tr)):.2f}")

rank correlation of the two score sets (same ordering of applicants = 100%): 99.5%


F1-best cutoff without SMOTE: 0.36
F1-best cutoff with SMOTE:    0.67


## 4. Test-set decisions (Results tab: confusion matrices and default rate among approved)

In [5]:
tp = pd.read_csv(DATA / "test_predictions.csv")
rows = {}
for s in ["Reference (rate + grade)", "Pipeline A (grade)", "Pipeline B (newcomer)"]:
    appr = tp[f"approved {s}"].astype(bool)
    d = tp["defaulted"].astype(bool)
    rows[s] = {"declined": int((~appr).sum()), "approved": int(appr.sum()),
               "defaulters caught": int((~appr & d).sum()), "good customers declined": int((~appr & ~d).sum()),
               "defaulters approved": int((appr & d).sum()),
               "recall": (~appr & d).sum() / d.sum(), "precision": (~appr & d).sum() / (~appr).sum(),
               "default rate among approved": (appr & d).sum() / appr.sum()}
print(f"test applicants {len(tp):,}; defaulters {int(tp['defaulted'].sum()):,}; "
      f"default rate with no model {tp['defaulted'].mean():.1%}")
pd.DataFrame(rows).T.style.format({"recall": "{:.1%}", "precision": "{:.1%}", "default rate among approved": "{:.1%}"})

test applicants 6,482; defaulters 1,418; default rate with no model 21.9%


,declined,approved,defaulters caught,good customers declined,defaulters approved,recall,precision,default rate among approved
Reference (rate + grade),1112.000000,5370.000000,1060.000000,52.000000,358.000000,74.8%,95.3%,6.7%
Pipeline A (grade),1102.000000,5380.000000,1054.000000,48.000000,364.000000,74.3%,95.6%,6.8%
Pipeline B (newcomer),1054.000000,5428.000000,864.000000,190.000000,554.000000,60.9%,82.0%,10.2%


## 5. Newcomer rates are pulled towards the middle (Result analysis finding 6)

Average error of Pipeline B's suggested rate (suggested − actual) for test applicants with a recorded rate.

In [6]:
r = tp[tp["loan_int_rate"].notna()]
err = r["suggested rate Pipeline B (newcomer)"] - r["loan_int_rate"]
print(f"actually charged under 8%: suggestion {err[r['loan_int_rate'] < 8].mean():+.1f} pp on average")
print(f"actually charged over 17%: suggestion {err[r['loan_int_rate'] > 17].mean():+.1f} pp on average")

actually charged under 8%: suggestion +3.1 pp on average
actually charged over 17%: suggestion -5.5 pp on average


## 6. Break-even interest rate (Limitations, point 8)

To break even on a loan, the interest earned from those who repay must cover the loss on those who default:
$(1-\text{PD})\cdot r \ge \text{PD}\cdot \text{LGD}$, so $r \ge \text{PD}\cdot\text{LGD}/(1-\text{PD})$, where LGD is the
share of the loan lost on default (before the lender's own funding cost). The highest rate in the data is shown for comparison.

In [7]:
be = pd.DataFrame({"PD": [0.05, 0.10, 0.20, 0.30, 0.37, 0.50]})
for lgd in (1.0, 0.6):
    be[f"break-even rate, lose {lgd:.0%}"] = be["PD"] * lgd / (1 - be["PD"])
print(f"highest interest rate in the data: {full['loan_int_rate'].max():.2f}%")
be.style.format({c: "{:.0%}" for c in be.columns})

highest interest rate in the data: 23.22%


,PD,"break-even rate, lose 100%","break-even rate, lose 60%"
0,5%,5%,3%
1,10%,11%,7%
2,20%,25%,15%
3,30%,43%,26%
4,37%,59%,35%
5,50%,100%,60%
